# Hansen Ch.16 Non-Stationary Time Series

**Chapter 16 Non-Stationary Time Series**

理论证明与**面向初学者的详细注释**见同目录 `Hansen_Ch16_Exercises_Solutions.md`（强烈建议先读 §0、§1）。

本 notebook：BN 数值核对 + FRED-MD 上 **ADF / KPSS / Johansen** 实证（16.12–16.14）+ 末尾的 **理论结论蒙特卡洛验证**。

> **写给只学过李子奈/陈强的同学：** 当序列**非平稳**（单位根 I(1)）时，第 14–15 章的渐近理论全部失效。三大灾难：
> - **伪回归**：两个独立 I(1) 回归，OLS $|t|>1.96$ 比例 **85.7%**（正态应仅 5%）！
> - **DF 非正态**：单位根下 $t$ 统计量服从 DF 分布（5% 临界值 $-2.86$ 而非 $-1.645$），用正态会**过度拒绝**。
> - **标准 CI 失效**：$\sqrt n$ 正态对称 CI 无效。
> 应对：ADF 检验（$H_0$ 单位根，用 DF 临界值）、KPSS（$H_0$ 平稳）、协整（$\beta'Y_t$ 平稳则不伪回归）、VECM。
> 核心逻辑：**"不拒绝单位根" ≠ "是单位根"**——检验只是证据不足。


In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from numpy.linalg import inv, det, eigvals
from scipy.linalg import toeplitz

ROOT = Path('/home/fang/Project/zhihu-paper/p1/hansen/econometrics/data')

# ---------- 16.1 核对 ----------
# S_t random walk: E=0, Var=t*sigma^2; Y_t=S_t/(sigma*sqrt(t)) not stationary
sigma = 1.0
t = np.arange(1, 6)
print('16.1 var[S_t] =', t * sigma**2)
print('Cov structure ratio sqrt(t/(t+k)) for t=10,k=5:', np.sqrt(10/15))

# ---------- 16.2 BN ----------
Theta1, Theta2 = 0.5, -0.2
C1 = 1 + Theta1 + Theta2
print('16.2 C(1) =', C1, '; C*(L) = -(%g) - (%g)L' % (Theta1+Theta2, Theta2))

# ---------- 16.5 cointegrating vector ----------
print('16.5 beta = (1, -1/2) or (2, -1)')


## ADF / KPSS / Johansen 工具函数

In [ ]:
def adf_regression(y, p, trend='c'):
    y = np.asarray(y, float)
    y = y[np.isfinite(y)]
    n0 = len(y)
    dy = np.diff(y)
    q = p - 1
    T = n0 - p
    if T < 10:
        return None
    Y = dy[p-1:]
    cols = [y[p-1:n0-1]]
    for j in range(1, q+1):
        cols.append(dy[p-1-j:n0-1-j])
    if trend in ('c', 'ct'):
        cols.append(np.ones(T))
    if trend == 'ct':
        cols.append(np.arange(p, n0, dtype=float))
    X = np.column_stack(cols)
    b = inv(X.T @ X) @ (X.T @ Y)
    e = Y - X @ b
    s2 = (e @ e) / (T - X.shape[1])
    se = np.sqrt(s2 * inv(X.T @ X)[0, 0])
    rho1 = b[0]
    s2ml = (e @ e) / T
    aic = np.log(s2ml) + 2 * X.shape[1] / T
    return dict(rho1=rho1, se=se, adf=rho1/se, n=T, p=p, aic=aic)


def select_adf(y, pmax=12, trend='ct'):
    best = None
    for p in range(1, pmax + 1):
        r = adf_regression(y, p, trend)
        if r is None:
            continue
        if best is None or r['aic'] < best['aic']:
            best = r
    return best


ADF_FULL = {
    'c':  [(1,-3.4),(2,-3.2),(3,-3.1),(4,-3.0),(5,-2.9),(7,-2.7),(10,-2.6),(15,-2.4),(20,-2.2),(30,-2.0),(50,-1.6),(70,-1.1),(90,-0.4)],
    'ct': [(1,-4.0),(2,-3.7),(3,-3.6),(4,-3.5),(5,-3.4),(7,-3.3),(10,-3.1),(15,-2.9),(20,-2.8),(30,-2.6),(50,-2.2),(70,-1.8),(90,-1.2)],
}


def adf_pvalue(stat, case='ct'):
    pts = ADF_FULL[case]
    if stat <= pts[0][1]:
        return pts[0][0] / 100
    if stat >= pts[-1][1]:
        return pts[-1][0] / 100
    for i in range(len(pts) - 1):
        p1, c1 = pts[i]
        p2, c2 = pts[i + 1]
        if c1 <= stat <= c2:
            return (p1 * (c2 - stat) + p2 * (stat - c1)) / (c2 - c1) / 100
    return np.nan


def kpss_stat(y, M, trend='c'):
    y = np.asarray(y, float)
    y = y[np.isfinite(y)]
    n = len(y)
    t = np.arange(1, n + 1, dtype=float)
    if trend == 'c':
        e = y - y.mean()
    else:
        X = np.column_stack([np.ones(n), t])
        b = inv(X.T @ X) @ (X.T @ y)
        e = y - X @ b
    S = np.cumsum(e)
    omega = np.mean(e ** 2)
    for ell in range(1, M + 1):
        w = 1 - ell / (M + 1)
        omega += 2 * w * np.mean(e[ell:] * e[:-ell])
    omega = max(omega, 1e-12)
    return np.sum(S ** 2) / (n ** 2 * omega), n


KPSS_FULL = {
    'c':  [(1,0.75),(2,0.62),(3,0.55),(4,0.50),(5,0.46),(7,0.41),(10,0.35),(15,0.28),(20,0.24),(30,0.18),(50,0.12),(70,0.08)],
    'ct': [(1,0.22),(2,0.19),(3,0.17),(4,0.16),(5,0.15),(7,0.13),(10,0.12),(15,0.10),(20,0.09),(30,0.08),(50,0.06),(70,0.04)],
}


def kpss_pvalue(stat, case='ct'):
    pts = KPSS_FULL[case]
    if stat >= pts[0][1]:
        return pts[0][0] / 100
    if stat <= pts[-1][1]:
        return 0.99
    for i in range(len(pts) - 1):
        p1, c1 = pts[i]
        p2, c2 = pts[i + 1]
        if c1 >= stat >= c2:
            return (p1 * (stat - c2) + p2 * (c1 - stat)) / (c1 - c2) / 100
    return np.nan


def johansen_trace(Y, p, trend=2):
    Y = np.asarray(Y, float)
    T0, m = Y.shape
    k = p - 1
    dY = np.diff(Y, axis=0)
    n = T0 - p
    Z0 = dY[p - 1:]
    Z1 = Y[p - 1:T0 - 1].copy()
    Z2_parts = []
    for j in range(1, k + 1):
        Z2_parts.append(dY[p - 1 - j:T0 - 1 - j])
    if trend == 2:
        Z1 = np.column_stack([Z1, np.ones(n)])
    elif trend == 3:
        Z2_parts.append(np.ones(n))
    if Z2_parts:
        Z2 = np.column_stack(Z2_parts)
        P = inv(Z2.T @ Z2)
        R0 = Z0 - Z2 @ (P @ (Z2.T @ Z0))
        R1 = Z1 - Z2 @ (P @ (Z2.T @ Z1))
    else:
        R0, R1 = Z0, Z1
    S00 = R0.T @ R0 / n
    S11 = R1.T @ R1 / n
    S01 = R0.T @ R1 / n
    A = inv(S11) @ S01.T @ inv(S00) @ S01
    ev = np.sort(np.real(eigvals(A)))[::-1][:m]
    LR = [-n * np.sum(np.log(np.maximum(1 - ev[r:m], 1e-12))) for r in range(m)]
    return dict(eigenvalues=ev, LR=LR, n=n, m=m)


def var_aic_p(Y, pmax=12):
    best_p, best_aic = 1, np.inf
    T0, m = Y.shape
    for p in range(1, pmax + 1):
        if T0 <= p + 5:
            continue
        y = Y[p:]
        parts = [np.ones((len(y), 1))]
        for j in range(1, p + 1):
            parts.append(Y[p - j:T0 - j])
        X = np.hstack(parts)
        B = inv(X.T @ X) @ (X.T @ y)
        E = y - X @ B
        Sig = E.T @ E / len(y)
        aic = np.log(max(det(Sig), 1e-30)) + 2 * (m * m * p + m) / len(y)
        if aic < best_aic:
            best_aic, best_p = aic, p
    return best_p


## 16.12–16.13 FRED-MD：ADF 与 KPSS

In [ ]:
md = pd.read_excel(ROOT / 'FRED-MD/FRED-MD.xlsx')
for c in md.columns:
    if c != 'time':
        md[c] = pd.to_numeric(md[c], errors='coerce')

series_spec = [
    ('log(rpi)',       lambda d: np.log(d['rpi'].dropna().values), 'ct'),
    ('indpro',         lambda d: d['indpro'].dropna().values, 'ct'),
    ('log(indpro)',    lambda d: np.log(d['indpro'].dropna().values), 'ct'),
    ('houst',          lambda d: d['houst'].dropna().values, 'c'),
    ('hwi',            lambda d: d['hwi'].dropna().values, 'ct'),
    ('log(clf16ov)',   lambda d: np.log(d['clf16ov'].dropna().values), 'ct'),
    ('claimsx',        lambda d: d['claimsx'].dropna().values, 'c'),
    ('log(ipfuels)',   lambda d: np.log(d['ipfuels'].dropna().values), 'ct'),
]

rows = []
for name, fn, trend in series_spec:
    y = fn(md)
    y = y[np.isfinite(y)]
    adf = select_adf(y, 12, trend)
    pv_a = adf_pvalue(adf['adf'], trend)
    M = int(np.ceil(3 * len(y) ** (1/3)))
    kp, n = kpss_stat(y, M, trend)
    pv_k = kpss_pvalue(kp, trend)
    rows.append({
        'series': name, 'trend': trend,
        'p_AR': adf['p'], 'rho-1': adf['rho1'], 'ADF': adf['adf'], 'ADF_p': pv_a,
        'M': M, 'KPSS': kp, 'KPSS_p': pv_k, 'n': n,
    })

res = pd.DataFrame(rows)
pd.set_option('display.float_format', lambda x: f'{x:0.4f}')
print(res.to_string(index=False))


## 16.14 Johansen 迹检验

In [ ]:
JOH_CV = {
    2: {1: {5: 9.2, 1: 12.7}, 2: {5: 20.3, 1: 25.1}},  # Trend Model 2
    3: {1: {5: 3.8, 1: 6.6}, 2: {5: 15.5, 1: 19.9}},  # Trend Model 3
}

pairs = [
    ('tb3ms, gs10', ['tb3ms', 'gs10'], False, 2),
    ('aaa, baa', ['aaa', 'baa'], False, 2),
    ('log(ipdcongd), log(ipncongd)', ['ipdcongd', 'ipncongd'], True, 3),
    ('log(ipdcongd), log(ipncongd) [Model2]', ['ipdcongd', 'ipncongd'], True, 2),
]

for label, cols, do_log, trend in pairs:
    df = md[cols].dropna()
    Y = df.values.astype(float)
    if do_log:
        Y = np.log(Y)
    p = var_aic_p(Y, 12)
    out = johansen_trace(Y, p, trend=trend)
    print(f'\n{label}')
    print(f'  VAR p={p}, trend model={trend}, n={out["n"]}')
    print(f'  eigenvalues: {np.round(out["eigenvalues"], 4)}')
    for r, lr in enumerate(out['LR']):
        d = out['m'] - r
        cv5 = JOH_CV[trend][d][5]
        cv1 = JOH_CV[trend][d][1]
        dec = 'reject 5%' if lr > cv5 else 'fail to reject'
        print(f'  LR({r})={lr:6.2f}  (m-r={d})  5%={cv5}  1%={cv1}  → {dec}')


### 实证要点

- **16.12–13：** 收入/劳动力/IP 类序列 ADF 不拒绝单位根且 KPSS 拒绝平稳 → 偏向 I(1)。
- **houst / claims：** ADF 拒绝单位根；KPSS 仅 borderline。
- **16.14：** 国债与信用利差月度序列迹检验支持 $r=1$；耐用品/非耐用品 IP 对数拒绝无协整。

## 理论结论的蒙特卡洛验证（无需外部数据）

以下单元格核对 ch16 的核心结论：(1) 随机游走 var(S_t)=tσ² 非平稳；(2) **伪回归**——两个独立 I(1) 回归有 85% "假显著"；(3) **DF 分布非正态**——单位根下 t 的 5% 分位 ≈ −1.93 而非 −1.645；(4) **协整**——公共趋势模型中 β'Y_t 平稳；(5) **过度差分**——Δ(e_t) 是 MA(1) 且 C(1)=0。可独立运行。

In [ ]:
import numpy as np
rng = np.random.default_rng(16)

# ===== 16.1: 随机游走 var(S_t) = tσ² (非平稳) =====
T, reps, sigma = 10000, 5000, 1.0
var_500 = [np.concatenate([[0], np.cumsum(rng.standard_normal(T))])[500]**2 for _ in range(reps)]
print(f"[16.1] var(S_500) MC={np.mean(var_500):.1f} 理论=500σ²={500*sigma**2}")
print(f"  Cov(Y_t,Y_(t+k))=sqrt(t/(t+k)) 依赖t ⇒ 非平稳")

# ===== 伪回归: 两个独立随机游走 OLS "显著"(应仅5%) =====
print(f"\n[伪回归] 两个独立 I(1) 的 OLS t 统计量:")
T, reps = 200, 10000
tstats = []
for _ in range(reps):
    X = np.cumsum(rng.standard_normal(T))
    Y = np.cumsum(rng.standard_normal(T))
    b = np.sum(X*Y) / np.sum(X**2)
    e = Y - X*b
    se = np.sqrt(np.sum(e**2) / (T-2) / np.sum((X - X.mean())**2))
    tstats.append(b / se)
tstats = np.array(tstats)
print(f"  |t|>1.96 比例={np.mean(np.abs(tstats)>1.96):.3f} (正态下应≈0.05! 这就是伪回归)")

# ===== DF 分布: 单位根下 t 非正态(偏左) =====
print(f"\n[DF 分布] 单位根下 α̂ 的 t 统计量(H₀:α=1)非正态:")
T, reps = 200, 20000
df_t = []
for _ in range(reps):
    Y = np.cumsum(rng.standard_normal(T))
    X, y = Y[:-1], Y[1:]
    b = np.sum(X*y) / np.sum(X**2)
    e = y - X*b
    s2 = np.sum(e**2) / (T-1)
    se = np.sqrt(s2 / np.sum(X**2))
    df_t.append((b - 1) / se)
df_t = np.array(df_t)
print(f"  DF t: 5%分位={np.percentile(df_t,5):.3f}, 1%分位={np.percentile(df_t,1):.3f}")
print(f"  (正态 5%=-1.645, 1%=-2.326; DF 更负 ⇒ 用正态临界值过度拒绝!)")

# ===== 16.5 协整: 公共趋势, β'Y_t 平稳 =====
print(f"\n[16.5 协整] 公共趋势 U_t, Y=U+v, X=2U+w, β=(1,-1/2):")
T = 100000
U = np.cumsum(rng.standard_normal(T))
v, w = rng.standard_normal(T), rng.standard_normal(T)
Y, X = U + v, 2*U + w
spread = Y - 0.5*X                                        # β'Y = v - 0.5w (平稳)
print(f"  Y_t I(1): ACF(1)={np.corrcoef(Y[:-1], Y[1:])[0,1]:.4f}")
print(f"  X_t I(1): ACF(1)={np.corrcoef(X[:-1], X[1:])[0,1]:.4f}")
print(f"  β'Y_t = Y-0.5X: ACF(1)={np.corrcoef(spread[:-1], spread[1:])[0,1]:.4f} (≈0 ⇒ 平稳!)")

# ===== 16.4 过度差分: Δ(e_t) 是 MA(1), C(1)=0 =====
print(f"\n[16.4 过度差分] 白噪声差分后:")
T = 200000
e = rng.standard_normal(T)
X = np.diff(e)                                            # X_t = e_t - e_{t-1}
g0 = np.var(X)
g1 = np.mean((X[1:] - X.mean()) * (X[:-1] - X.mean()))
print(f"  γ(0)={g0:.3f}(理论2σ²=2), γ(1)={g1:.3f}(理论-σ²=-1), ρ(1)={g1/g0:.3f}(理论-0.5)")
print(f"  C(1)=1-1=0 ⇒ 长期方差=2+2(-1)=0 ⇒ 平稳但非I(0) (过度差分)")
